# Qwen2.5-VL Prompt Tuning with PEFT on RSICD
This notebook fine-tunes a soft prompt for Qwen2.5-VL 7B on the RSICD dataset using 🤗 PEFT.

In [ ]:
!pip install -q torch transformers peft datasets evaluate accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import PromptTuningConfig, get_peft_model, TaskType
import evaluate
import time

## Configurations

In [ ]:
MODEL_NAME = "Qwen/Qwen-VL-7B"
DATASET_NAME = "arampacha/rsicd"
MAX_LENGTH = 128
PROMPT_LENGTH = 20
BATCH_SIZE = 1
NUM_TRAIN_EPOCHS = 3
OUTPUT_DIR = "./qwen_rsicd_output"
device = "cuda" if torch.cuda.is_available() else "cpu" 

## Load Dataset

In [ ]:
dataset = load_dataset(DATASET_NAME, split="train[:500]")

## Load Model and Processor

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")

## Apply Prompt Tuning using PEFT

In [ ]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=PROMPT_LENGTH,
    tokenizer_name_or_path=MODEL_NAME,
)

model = get_peft_model(model, peft_config)

## Preprocess Dataset

In [ ]:
def preprocess(example):
    image = example["image"]
    prompt = "Describe the satellite image."
    encoding = processor(text=prompt, images=image, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LENGTH)
    labels = processor.tokenizer(example["captions"][0], padding="max_length", truncation=True, max_length=MAX_LENGTH, return_tensors="pt").input_ids
    encoding["labels"] = labels.squeeze()
    return encoding

dataset = dataset.map(preprocess)
dataset.set_format(type="torch")

## Training Arguments

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    fp16=True,
    report_to="none"
)

## Evaluation Metrics

In [ ]:
cider = evaluate.load("cider")
spice = evaluate.load("spice")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    decoded_preds = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {
        "CIDEr": cider.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"],
        "SPICE": spice.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"]
    }

## Train the Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

## Inference Time and VRAM Usage

In [ ]:
torch.cuda.reset_peak_memory_stats()
start = time.time()
sample = dataset[0]
out = model.generate(**processor(text="Describe this image.", images=sample["image"], return_tensors="pt").to(device))
end = time.time()
print("Generated:", processor.tokenizer.decode(out[0], skip_special_tokens=True))
print("Inference Time:", end - start, "seconds")
print("Peak VRAM:", torch.cuda.max_memory_allocated() / 1e9, "GB")